In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

from io import BytesIO
from scipy.optimize import curve_fit
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.pipeline import make_pipeline

from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    Image as RLImage, PageBreak, HRFlowable
)

from pybaseball import batting_stats, statcast_sprint_speed

In [2]:
# =============================================================================
# DATA IMPORT  —  TRAIN on 2019-2024,  VALIDATE on 2025
# =============================================================================

TRAIN_YEARS = range(2019, 2025)   # 2019 – 2024 inclusive
VAL_YEAR    = 2025                 # held-out year for out-of-sample validation

def load_batting_sprint(years, qual=100):
    """Pull batting stats + sprint speed for a list of years and merge them."""
    bat_frames = []
    for year in years:
        df = batting_stats(year, qual=qual)
        df['Season'] = year
        bat_frames.append(df)
    batting = pd.concat(bat_frames, ignore_index=True)

    sp_frames = []
    for year in years:
        df = statcast_sprint_speed(year)
        df['year'] = year
        sp_frames.append(df)
    sprint = pd.concat(sp_frames, ignore_index=True)

    sprint['last_name, first_name'] = sprint['last_name, first_name'].apply(
        lambda x: ' '.join(reversed(x.split(', ')))
        if isinstance(x, str) and ', ' in x else x
    )
    merged = pd.merge(
        sprint, batting,
        left_on=['last_name, first_name', 'year'],
        right_on=['Name', 'Season'],
        how='left'
    )

    merged['TB'] = (
        (merged['H'] - merged['2B'] - merged['3B'] - merged['HR'])
        + merged['2B'] * 2 + merged['3B'] * 3 + merged['HR'] * 4
    )
    merged['MB_Value'] = (
        ((merged['H'] + merged['BB'] - merged['CS'])
         * (merged['TB'] + 0.7 * merged['SB']))
        / (merged['AB'] + merged['BB'] + merged['CS'])
    )
    return merged

print("Loading TRAINING data (2019–2024)…")
train_df = load_batting_sprint(TRAIN_YEARS)
train_df.to_csv('Train_2019_2024.csv', index=False)

print("Loading VALIDATION data (2025)…")
val_df = load_batting_sprint([VAL_YEAR])
val_df.to_csv('Val_2025.csv', index=False)

print(f"  Train rows: {len(train_df)}   |   Val rows: {len(val_df)}")

Loading TRAINING data (2019–2024)…
Loading VALIDATION data (2025)…
  Train rows: 3326   |   Val rows: 581


In [3]:
# =============================================================================
# CONFIG
# =============================================================================

BASE_FEATURES = [
    'O-Swing%', 'Z-Swing%', 'Swing%', 'O-Contact%', 'Z-Contact%', 'Contact%',
    'Zone%', 'F-Strike%', 'SwStr%', 'Barrel%', 'maxEV', 'HardHit', 'HardHit%',
    'CStr%', 'CSW%', 'EV', 'LA', 'bolts', 'hp_to_1b', 'sprint_speed', 'Pitches',
    'LD%', 'GB%'
]

REGRESSION_TARGETS = ['TB', 'BB', 'AB', 'H']
SB_CS_TARGETS      = ['SB', 'CS']
ALL_TARGETS        = REGRESSION_TARGETS + SB_CS_TARGETS
R2_THRESHOLD       = 0.8   # still used to decide *which* model to deploy
XMB_R2_THRESHOLD   = 0.5   # minimum R² for a component stat to use x<stat> in xMB formula
                            # if a component's best R² < this, the real observed stat is used instead

PALETTE = {
    'TB': '#2980B9', 'BB': '#27AE60', 'AB': '#8E44AD', 'H': '#E67E22',
    'SB': '#E74C3C', 'CS': '#1ABC9C'
}
MODEL_COLORS = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628", "#1a9850"]

In [4]:
# =============================================================================
# UTILITIES
# =============================================================================

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

def adj_r2(r2, n, k):
    if n - k - 1 <= 0:
        return np.nan
    return round(1 - (1 - r2) * (n - 1) / (n - k - 1), 4)

def top_corr_features(data, target, features, top_n=5, min_abs=0.05):
    avail = [f for f in features if f in data.columns]
    corr = data[avail + [target]].corr()[target].drop(target).dropna()
    corr = corr[corr.abs() >= min_abs].abs().sort_values(ascending=False)
    return corr.index[:top_n].tolist()

def rl_image(buf, width_in, fig_w=7, fig_h=4.5):
    buf.seek(0)
    img = RLImage(buf, width=width_in * inch, height=width_in * inch * (fig_h / fig_w))
    img.hAlign = "CENTER"
    return img

def divider():
    return HRFlowable(width="100%", thickness=1, color=colors.HexColor("#cccccc"),
                      spaceAfter=8, spaceBefore=4)

def th(text, S):
    return Paragraph(text, S["header"])

def styled_table(rows, col_widths, header_color="#2C3E50", highlight_row=None):
    t = Table(rows, colWidths=col_widths, repeatRows=1)
    cmds = [
        ("BACKGROUND",     (0, 0),  (-1, 0),  colors.HexColor(header_color)),
        ("TEXTCOLOR",      (0, 0),  (-1, 0),  colors.white),
        ("FONTSIZE",       (0, 0),  (-1, -1), 8.5),
        ("ALIGN",          (0, 0),  (-1, -1), "CENTER"),
        ("ROWBACKGROUNDS", (0, 1),  (-1, -1), [colors.HexColor("#ECF0F1"), colors.white]),
        ("GRID",           (0, 0),  (-1, -1), 0.5, colors.grey),
        ("TOPPADDING",     (0, 0),  (-1, -1), 4),
        ("BOTTOMPADDING",  (0, 0),  (-1, -1), 4),
        ("LEFTPADDING",    (0, 0),  (-1, -1), 6),
        ("RIGHTPADDING",   (0, 0),  (-1, -1), 6),
    ]
    if highlight_row is not None:
        cmds += [
            ("BACKGROUND", (0, highlight_row), (-1, highlight_row), colors.HexColor("#d4edda")),
            ("FONTNAME",   (0, highlight_row), (-1, highlight_row), "Helvetica-Bold"),
        ]
    t.setStyle(TableStyle(cmds))
    return t

In [5]:
# =============================================================================
# REGRESSION MODELS  (trained on 2019-2024 only)
# =============================================================================

def run_regression_analysis(target, features, data):
    """Fit models on `data` (training set). Return fitted objects + metadata."""
    print(f"\n{'='*60}\n  TARGET: {target}\n{'='*60}")

    # Correlation-filtered features for Linear & Ridge
    cols = features + [target]
    corr = data[cols].corr()[target]
    selected = corr[(corr.abs() > 0.3) & (corr.index != target)].index.tolist()
    print("Selected features (Linear/Ridge):", selected)

    # All available features for Lasso & Random Forest
    all_features = [f for f in features if f in data.columns]
    print("All features (Lasso/RF):", all_features)

    result = {
        'target': target,
        'selected_features': selected,
        'all_features': all_features,
        'models': {},
        'fitted_objects': {},   # store fitted model + scaler for 2025 deployment
    }
    if not selected:
        print(f"  No features with |r| > 0.3 for {target}. Skipping.")
        return result

    # ── Data splits for Linear / Ridge (correlation-filtered features) ──
    X_red = data[selected].dropna()
    y_red = data.loc[X_red.index, target]
    X_train_red, X_test_red, y_train_red, y_test_red = train_test_split(
        X_red, y_red, test_size=0.2, random_state=42)

    scaler_red = StandardScaler()
    Xtr_red_s  = scaler_red.fit_transform(X_train_red)
    Xte_red_s  = scaler_red.transform(X_test_red)
    X_red_s    = scaler_red.fit_transform(X_red)

    # ── Data splits for Lasso / Random Forest (all features) ──
    X_all = data[all_features].dropna()
    y_all = data.loc[X_all.index, target]
    X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
        X_all, y_all, test_size=0.2, random_state=42)

    scaler_all = StandardScaler()
    Xtr_all_s  = scaler_all.fit_transform(X_train_all)
    Xte_all_s  = scaler_all.transform(X_test_all)
    X_all_s    = scaler_all.fit_transform(X_all)

    # Linear (correlation-filtered features only)
    lr = LinearRegression().fit(Xtr_red_s, y_train_red)
    lp = lr.predict(Xte_red_s)
    lc = cross_val_score(lr, X_red_s, y_red, cv=5, scoring='r2')
    result['models']['Linear Regression'] = {
        'r2': r2_score(y_test_red, lp), 'rmse': rmse(y_test_red, lp),
        'cv_r2_mean': lc.mean(), 'cv_r2_std': lc.std()
    }

    # Ridge (correlation-filtered features only)
    rr = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5).fit(Xtr_red_s, y_train_red)
    rp = rr.predict(Xte_red_s)
    rc = cross_val_score(rr, X_red_s, y_red, cv=5, scoring='r2')
    result['models']['Ridge Regression'] = {
        'r2': r2_score(y_test_red, rp), 'rmse': rmse(y_test_red, rp),
        'cv_r2_mean': rc.mean(), 'cv_r2_std': rc.std(),
        'best_alpha': rr.alpha_
    }

    # Lasso (all features — lets it do its own variable selection)
    la = LassoCV(cv=5, random_state=42, max_iter=10000).fit(Xtr_all_s, y_train_all)
    lap = la.predict(Xte_all_s)
    lac = cross_val_score(la, X_all_s, y_all, cv=5, scoring='r2')
    result['models']['Lasso Regression'] = {
        'r2': r2_score(y_test_all, lap), 'rmse': rmse(y_test_all, lap),
        'cv_r2_mean': lac.mean(), 'cv_r2_std': lac.std(),
        'best_alpha': la.alpha_,
        'coefficients': dict(zip(all_features, la.coef_))
    }

    # Random Forest (all features — handles irrelevant ones naturally)
    rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train_all, y_train_all)
    rfp = rf.predict(X_test_all)
    rfc = cross_val_score(rf, X_all, y_all, cv=5, scoring='r2')
    result['models']['Random Forest'] = {
        'r2': r2_score(y_test_all, rfp), 'rmse': rmse(y_test_all, rfp),
        'cv_r2_mean': rfc.mean(), 'cv_r2_std': rfc.std(),
        'feature_importances': dict(zip(all_features, rf.feature_importances_))
    }

    # ── Refit the BEST model on the FULL training set so it's ready for 2025 ──
    best_name = max(result['models'], key=lambda k: result['models'][k]['r2'])

    # Use the correct feature set depending on which model won
    if best_name in ('Linear Regression', 'Ridge Regression'):
        best_features = selected
    else:
        best_features = all_features

    full_clean = data[best_features + [target]].dropna()
    Xfull = full_clean[best_features].values
    yfull = full_clean[target].values

    full_scaler = StandardScaler()
    Xfull_s = full_scaler.fit_transform(Xfull)

    if best_name == "Linear Regression":
        best_fitted = LinearRegression().fit(Xfull_s, yfull)
    elif best_name == "Ridge Regression":
        best_fitted = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0], cv=5).fit(Xfull_s, yfull)
    elif best_name == "Lasso Regression":
        best_fitted = LassoCV(cv=5, random_state=42, max_iter=10000).fit(Xfull_s, yfull)
    elif best_name == "Random Forest":
        best_fitted = RandomForestRegressor(n_estimators=100, random_state=42).fit(Xfull, yfull)
    else:
        best_fitted = None

    result['fitted_objects'] = {
        'best_name':     best_name,
        'model':         best_fitted,
        'scaler':        full_scaler,
        'uses_scaled':   best_name != "Random Forest",
        'best_features': best_features,  # track which feature set the deployed model needs
    }

    for name, m in result['models'].items():
        print(f"  {name:<25} R²={m['r2']:.4f}  CV R²={m['cv_r2_mean']:.4f}  RMSE={m['rmse']:.4f}")

    print(f"  → Deployed: '{best_name}'")
    return result

In [6]:
# =============================================================================
# SB / CS MODELS  (trained on 2019-2024 only)
# =============================================================================

def fit_sb_cs_models(data, target):
    best_feats = top_corr_features(data, target, BASE_FEATURES)
    if not best_feats:
        print(f"  No correlated features for {target}.")
        return [], None, None, None, None, None

    primary = best_feats[0]
    print(f"  {target}: primary={primary}, features={best_feats}")

    clean = data[[primary, target]].dropna()
    x = clean[primary].values
    y = clean[target].values.astype(float)
    x_plot = np.linspace(x.min(), x.max(), 300)

    results, curves = [], {}

    # 1. Linear
    lm = LinearRegression().fit(x.reshape(-1, 1), y)
    yp = lm.predict(x.reshape(-1, 1))
    r2 = r2_score(y, yp)
    results.append({"Model": "Linear", "R2": round(r2, 4), "AdjR2": adj_r2(r2, len(y), 1),
                    "RMSE": round(rmse(y, yp), 4), "Params": 1,
                    "Notes": "Baseline", "y_pred": yp})
    curves["Linear"] = lm.predict(x_plot.reshape(-1, 1))

    # 2. Sqrt
    xs = np.sqrt(np.maximum(x, 0))
    lm_sq = LinearRegression().fit(xs.reshape(-1, 1), y)
    yp = lm_sq.predict(xs.reshape(-1, 1))
    r2 = r2_score(y, yp)
    results.append({"Model": "Sqrt Transform", "R2": round(r2, 4), "AdjR2": adj_r2(r2, len(y), 1),
                    "RMSE": round(rmse(y, yp), 4), "Params": 1,
                    "Notes": f"{target} ~ sqrt({primary})", "y_pred": yp})
    curves["Sqrt Transform"] = lm_sq.predict(np.sqrt(np.maximum(x_plot, 0)).reshape(-1, 1))

    # 3. Log
    xl = np.log(np.maximum(x, 0.01))
    lm_log = LinearRegression().fit(xl.reshape(-1, 1), y)
    yp = lm_log.predict(xl.reshape(-1, 1))
    r2 = r2_score(y, yp)
    results.append({"Model": "Log Transform", "R2": round(r2, 4), "AdjR2": adj_r2(r2, len(y), 1),
                    "RMSE": round(rmse(y, yp), 4), "Params": 1,
                    "Notes": f"{target} ~ log({primary})", "y_pred": yp})
    curves["Log Transform"] = lm_log.predict(np.log(np.maximum(x_plot, 0.01)).reshape(-1, 1))

    # 4. Polynomial deg-2
    poly2 = make_pipeline(PolynomialFeatures(2), LinearRegression())
    poly2.fit(x.reshape(-1, 1), y)
    yp = poly2.predict(x.reshape(-1, 1))
    r2 = r2_score(y, yp)
    results.append({"Model": "Polynomial deg-2", "R2": round(r2, 4), "AdjR2": adj_r2(r2, len(y), 2),
                    "RMSE": round(rmse(y, yp), 4), "Params": 2,
                    "Notes": "Quadratic", "y_pred": yp})
    curves["Polynomial deg-2"] = poly2.predict(x_plot.reshape(-1, 1))

    # 5. Power Law / Neg Exponential
    if target == "SB":
        try:
            def power_law(xv, a, b): return a * np.power(np.maximum(xv, 0.01), b)
            popt, _ = curve_fit(power_law, x, y, p0=[1, 0.5], maxfev=8000)
            yp = power_law(x, *popt)
            r2 = r2_score(y, yp)
            results.append({"Model": "Power Law (a·x^b)", "R2": round(r2, 4),
                            "AdjR2": adj_r2(r2, len(y), 2), "RMSE": round(rmse(y, yp), 4),
                            "Params": 2, "Notes": f"a={popt[0]:.3f}, b={popt[1]:.3f}",
                            "y_pred": yp, "_popt": popt})
            curves["Power Law (a·x^b)"] = power_law(x_plot, *popt)
        except Exception:
            pass
    else:
        try:
            def neg_exp(xv, a, b): return a * np.exp(b * xv)
            popt, _ = curve_fit(neg_exp, x, y, p0=[10, -1], maxfev=8000)
            yp = neg_exp(x, *popt)
            r2 = r2_score(y, yp)
            results.append({"Model": "Neg. Exponential (a·e^bx)", "R2": round(r2, 4),
                            "AdjR2": adj_r2(r2, len(y), 2), "RMSE": round(rmse(y, yp), 4),
                            "Params": 2, "Notes": f"a={popt[0]:.3f}, b={popt[1]:.3f}",
                            "y_pred": yp, "_popt": popt})
            curves["Neg. Exponential (a·e^bx)"] = neg_exp(x_plot, *popt)
        except Exception:
            pass

    # 6. GLM
    glm_label = "Poisson GLM" if target == "SB" else "Neg. Binomial GLM"
    glm_result = None
    try:
        clean_m = data[best_feats + [target]].dropna()
        y_m     = clean_m[target].values.astype(float)
        X_raw   = clean_m[best_feats].values.astype(float)
        scaler_m = StandardScaler()
        X_m     = np.column_stack([np.ones(len(X_raw)), scaler_m.fit_transform(X_raw)])
        beta    = np.zeros(X_m.shape[1])
        for _ in range(100):
            mu = np.clip(np.exp(X_m @ beta), 1e-10, None)
            W  = np.diag(mu)
            z  = X_m @ beta + (y_m - mu) / mu
            try:
                beta = np.linalg.solve(X_m.T @ W @ X_m, X_m.T @ W @ z)
            except np.linalg.LinAlgError:
                break
        yp_m = np.exp(X_m @ beta)
        r2_m = r2_score(y_m, yp_m)
        glm_result = {
            "Model": f"{glm_label} ({len(best_feats)} features)",
            "R2": round(r2_m, 4), "AdjR2": adj_r2(r2_m, len(y_m), len(best_feats)),
            "RMSE": round(rmse(y_m, yp_m), 4), "Params": len(best_feats),
            "Notes": "Count GLM — log-link, multi-feature",
            "coefficients": dict(zip(["const"] + best_feats, beta)),
            "_scaler": scaler_m,
            "_beta":   beta,
            "_features": best_feats,
        }
        results.append(glm_result)
    except Exception as e:
        results.append({"Model": glm_label, "R2": np.nan, "AdjR2": np.nan,
                        "RMSE": np.nan, "Params": 0, "Notes": f"Failed: {e}"})

    best = max((r for r in results if not np.isnan(r.get("R2", np.nan))), key=lambda r: r["R2"])
    print(f"  {target} best: {best['Model']} (R²={best['R2']})")
    return results, primary, x, y, x_plot, curves


def make_scatter_plot(x, y, results, x_plot, curves, primary, target):
    color = PALETTE[target]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(x, y, alpha=0.25, s=15, color=color, label="Observations")
    valid_r2 = {r["Model"]: r["R2"] for r in results if not np.isnan(r.get("R2", np.nan))}
    for i, (name, curve) in enumerate(curves.items()):
        r2 = valid_r2.get(name)
        lbl = f"{name} (R²={r2})" if r2 is not None else name
        ax.plot(x_plot, curve, color=MODEL_COLORS[i % len(MODEL_COLORS)], lw=2, label=lbl)
    ax.set_xlabel(primary, fontsize=10)
    ax.set_ylabel(target, fontsize=10)
    ax.set_title(f"{target} vs {primary} — Model Fits (Train 2019-2024)", fontsize=12,
                 fontweight="bold", color=color)
    ax.legend(fontsize=7.5, loc="best")
    ax.grid(True, alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150)
    plt.close(fig)
    return buf


def make_residual_plot(x, y, results, primary, target):
    valid = [r for r in results if not np.isnan(r.get("R2", np.nan)) and "y_pred" in r]
    if not valid:
        return None
    cols = 3
    rows = (len(valid) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(9, rows * 2.8))
    axes = axes.flatten()
    for i, r in enumerate(valid):
        ax = axes[i]
        ax.scatter(x, r["y_pred"] - y, alpha=0.25, s=10, color=MODEL_COLORS[i % len(MODEL_COLORS)])
        ax.axhline(0, color="black", lw=1.2, linestyle="--")
        ax.set_title(r["Model"], fontsize=8, fontweight="bold")
        ax.set_xlabel(primary, fontsize=7)
        ax.set_ylabel("Residual", fontsize=7)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    fig.suptitle(f"{target} — Residual Plots (Train 2019-2024)", fontsize=10, fontweight="bold")
    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return buf

In [7]:
# =============================================================================
# 2025 DEPLOYMENT  —  predict each target using models trained on 2019-2024
# =============================================================================

def deploy_to_2025(df_2025, all_results_reg, all_results_sb_cs):
    """
    Apply the trained models to df_2025 and add x<Target> columns.
    The actual 2025 values are left untouched for comparison.
    """
    result_df = df_2025.copy()

    # ── TB, BB, AB, H ──────────────────────────────────────────────────────
    for res in all_results_reg:
        target   = res['target']
        col_name = f"x{target}"
        fo       = res.get('fitted_objects', {})
        model    = fo.get('model')
        scaler   = fo.get('scaler')
        # Use best_features if available (respects which feature set the deployed model was trained on)
        selected = fo.get('best_features', res['selected_features'])

        if model is None or not selected:
            print(f"  [{target}] No fitted model — using actual.")
            result_df[col_name] = result_df.get(target, np.nan)
            continue

        avail = [f for f in selected if f in result_df.columns]
        if not avail:
            print(f"  [{target}] Features missing in 2025 data — using actual.")
            result_df[col_name] = result_df.get(target, np.nan)
            continue

        X_raw = result_df[avail].values.astype(float)
        has   = ~np.isnan(X_raw).any(axis=1)
        preds = np.full(len(result_df), np.nan)

        try:
            if fo['uses_scaled']:
                X_proc = scaler.transform(X_raw)
                preds[has] = model.predict(X_proc[has]).clip(0)
            else:  # Random Forest
                preds[has] = model.predict(X_raw[has]).clip(0)
        except Exception as e:
            print(f"  [{target}] Prediction error: {e}")

        result_df[col_name] = np.where(np.isnan(preds), result_df.get(target, np.nan), preds)
        best_name = fo.get('best_name', '?')
        print(f"  [{target}] Deployed '{best_name}' → '{col_name}'")

    # ── SB, CS ─────────────────────────────────────────────────────────────
    for target in SB_CS_TARGETS:
        col_name = f"x{target}"
        data     = all_results_sb_cs.get(target, {})
        primary  = data.get('primary')
        results  = data.get('results', [])

        valid = [r for r in results if not np.isnan(r.get('R2', np.nan))]
        if not valid:
            result_df[col_name] = result_df.get(target, np.nan)
            continue

        best = max(valid, key=lambda r: r['R2'])
        model_name = best['Model']
        print(f"  [{target}] Deploying '{model_name}' (train R²={best['R2']}) → '{col_name}'")

        # GLM path
        if "GLM" in model_name and "_beta" in best:
            features_used = best['_features']
            avail         = [f for f in features_used if f in result_df.columns]
            if avail:
                X_raw    = result_df[avail].values.astype(float)
                X_scaled = best['_scaler'].transform(X_raw)
                beta     = best['_beta']
                X_design = np.hstack([np.ones((len(X_scaled), 1)), X_scaled])
                result_df[col_name] = np.exp(X_design @ beta).clip(0)
            else:
                result_df[col_name] = result_df.get(target, np.nan)
            continue

        # Univariate path
        if primary is None or primary not in result_df.columns:
            result_df[col_name] = result_df.get(target, np.nan)
            continue

        x_vals  = result_df[primary].values.astype(float)
        x_train = train_df[primary].dropna().values
        y_train = train_df.loc[train_df[primary].notna(), target].values.astype(float)

        try:
            if model_name == "Linear":
                m_c, b_c = np.polyfit(x_train, y_train, 1)
                result_df[col_name] = (m_c * x_vals + b_c).clip(0)
            elif model_name == "Sqrt Transform":
                lm = LinearRegression().fit(np.sqrt(np.maximum(x_train, 0)).reshape(-1, 1), y_train)
                result_df[col_name] = lm.predict(
                    np.sqrt(np.maximum(x_vals, 0)).reshape(-1, 1)).clip(0)
            elif model_name == "Log Transform":
                lm = LinearRegression().fit(np.log(np.maximum(x_train, 0.01)).reshape(-1, 1), y_train)
                result_df[col_name] = lm.predict(
                    np.log(np.maximum(x_vals, 0.01)).reshape(-1, 1)).clip(0)
            elif model_name == "Polynomial deg-2":
                poly = make_pipeline(PolynomialFeatures(2), LinearRegression())
                poly.fit(x_train.reshape(-1, 1), y_train)
                result_df[col_name] = poly.predict(x_vals.reshape(-1, 1)).clip(0)
            elif "Power Law" in model_name and "_popt" in best:
                popt = best['_popt']
                def power_law(xv, a, b): return a * np.power(np.maximum(xv, 0.01), b)
                result_df[col_name] = power_law(x_vals, *popt).clip(0)
            elif "Exponential" in model_name and "_popt" in best:
                popt = best['_popt']
                def neg_exp(xv, a, b): return a * np.exp(b * xv)
                result_df[col_name] = neg_exp(x_vals, *popt).clip(0)
            else:
                result_df[col_name] = result_df.get(target, np.nan)
        except Exception as e:
            print(f"    WARNING: {e}. Falling back to actual.")
            result_df[col_name] = result_df.get(target, np.nan)

    return result_df

In [8]:
# =============================================================================
# MB / xMB
# =============================================================================

# The six component stats used in the MB formula
XMB_COMPONENTS = ['H', 'BB', 'CS', 'TB', 'SB', 'AB']

def get_component_r2(target, all_results_reg, all_results_sb_cs):
    """Return the best training R² achieved for a given target stat."""
    # Regression targets (TB, BB, AB, H)
    for res in all_results_reg:
        if res['target'] == target:
            if not res['models']:
                return 0.0
            return max(m['r2'] for m in res['models'].values())
    # SB / CS targets
    if target in all_results_sb_cs:
        data  = all_results_sb_cs[target]
        valid = [r for r in data.get('results', []) if not np.isnan(r.get('R2', np.nan))]
        if not valid:
            return 0.0
        return max(r['R2'] for r in valid)
    return 0.0


def select_xmb_components(all_results_reg, all_results_sb_cs, threshold=XMB_R2_THRESHOLD):
    """
    For each of the 6 MB component stats, decide whether to use the expected (x<stat>)
    or the real observed stat based on the model's training R².

    Returns
    -------
    component_map : dict  {stat: ('x<stat>' | stat, r2)}
    """
    component_map = {}
    for stat in XMB_COMPONENTS:
        r2 = get_component_r2(stat, all_results_reg, all_results_sb_cs)
        if r2 >= threshold:
            use_col = f"x{stat}"
            note    = f"x{stat}  (R²={r2:.4f} ≥ {threshold})"
        else:
            use_col = stat
            note    = f"{stat}  (R²={r2:.4f} < {threshold}, using actual)"
        component_map[stat] = {'col': use_col, 'r2': r2, 'note': note}
        print(f"  xMB component [{stat}]: {note}")
    return component_map


def compute_mb(df, component_map=None):
    """
    Compute MB (always from real stats) and xMB (using x<stat> or real stat
    per component, based on component_map).

    If component_map is None, xMB falls back to using all x<stat> columns
    (original behaviour).
    """
    # Real MB — always from observed stats
    df["MB"] = round(
        ((df["H"] + df["BB"] - df["CS"]) * (df["TB"] + 0.7 * df["SB"]))
        / (df["AB"] + df["BB"] + df["CS"])
    )

    if component_map is None:
        # Fallback: use all x-columns (original behaviour)
        h_col  = "xH";  bb_col = "xBB"; cs_col = "xCS"
        tb_col = "xTB"; sb_col = "xSB"; ab_col = "xAB"
    else:
        h_col  = component_map['H']['col']
        bb_col = component_map['BB']['col']
        cs_col = component_map['CS']['col']
        tb_col = component_map['TB']['col']
        sb_col = component_map['SB']['col']
        ab_col = component_map['AB']['col']

    print("\n  xMB formula being applied:")
    print(f"    xMB = (({h_col} + {bb_col} - {cs_col}) * ({tb_col} + 0.7 * {sb_col}))")
    print(f"          / ({ab_col} + {bb_col} + {cs_col})")
    mixed = any(
        col == stat
        for stat, col in [(h_col,'H'),(bb_col,'BB'),(cs_col,'CS'),
                          (tb_col,'TB'),(sb_col,'SB'),(ab_col,'AB')]
    )
    if mixed:
        print("  ⚠  Some components use ACTUAL stats (R² below threshold).")
    else:
        print("  ✓  All components use expected (x<stat>) values.")

    df["xMB"] = round(
        ((df[h_col] + df[bb_col] - df[cs_col]) * (df[tb_col] + 0.7 * df[sb_col]))
        / (df[ab_col] + df[bb_col] + df[cs_col])
    )
    return df

In [9]:
# =============================================================================
# 2025 OUT-OF-SAMPLE VALIDATION  (the new core analysis)
# =============================================================================

def run_2025_validation(val_pred_df):
    """
    Compare predicted (x<Target>) vs actual (<Target>) for 2025.
    Returns a dict of per-target metrics and combined clean DataFrame.
    """
    all_metrics = {}
    all_targets = REGRESSION_TARGETS + SB_CS_TARGETS + ['MB']

    for target in all_targets:
        xcol = f"x{target}"
        if target not in val_pred_df.columns or xcol not in val_pred_df.columns:
            continue

        clean = val_pred_df[[target, xcol]].dropna()
        if len(clean) < 5:
            continue

        actual = clean[target].values.astype(float)
        pred   = clean[xcol].values.astype(float)

        pe  = (pred - actual) / np.where(np.abs(actual) < 0.01, np.nan, actual) * 100
        ape = np.abs(pe)

        all_metrics[target] = {
            'n':             len(clean),
            'r2':            round(r2_score(actual, pred), 4),
            'mae':           round(mae(actual, pred), 3),
            'rmse':          round(rmse(actual, pred), 3),
            'mean_pe':       round(np.nanmean(pe),  2),
            'median_pe':     round(np.nanmedian(pe),2),
            'mean_ape':      round(np.nanmean(ape), 2),
            'pct_within_5':  round((ape <= 5).mean()  * 100, 1),
            'pct_within_10': round((ape <= 10).mean() * 100, 1),
            'pct_within_20': round((ape <= 20).mean() * 100, 1),
        }
        print(f"  {target:<4}  R²={all_metrics[target]['r2']:.4f}  "
              f"MAE={all_metrics[target]['mae']:.3f}  "
              f"RMSE={all_metrics[target]['rmse']:.3f}  "
              f"MeanAPE={all_metrics[target]['mean_ape']:.1f}%")

    return all_metrics


def fig_2025_scatter(val_pred_df, target):
    """Actual vs Predicted scatter for one target on 2025 data."""
    xcol  = f"x{target}"
    clean = val_pred_df[[target, xcol, 'Name']].dropna()
    if len(clean) < 5:
        return None

    actual = clean[target].values.astype(float)
    pred   = clean[xcol].values.astype(float)
    r2     = r2_score(actual, pred)

    # OLS fit line
    lm    = LinearRegression().fit(pred.reshape(-1, 1), actual)
    x_line = np.linspace(pred.min(), pred.max(), 200)
    y_line = lm.predict(x_line.reshape(-1, 1))

    color = PALETTE.get(target, '#333333')
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(pred, actual, alpha=0.35, s=20, color=color, label="Players")
    ax.plot(x_line, y_line,  color="#E74C3C",  lw=2,  label=f"OLS fit (R²={r2:.3f})")
    ax.plot(x_line, x_line,  color="grey",     lw=1.2, linestyle="--", label="Perfect (y=x)")
    ax.set_xlabel(f"Predicted {target}  (x{target})", fontsize=10)
    ax.set_ylabel(f"Actual {target}  (2025)", fontsize=10)
    ax.set_title(f"2025 Out-of-Sample: {target} vs x{target}", fontsize=11,
                 fontweight="bold", color=color)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return buf


def fig_2025_residuals(val_pred_df, target):
    """Residual (Predicted − Actual) vs Actual for 2025."""
    xcol  = f"x{target}"
    clean = val_pred_df[[target, xcol]].dropna()
    if len(clean) < 5:
        return None

    actual = clean[target].values.astype(float)
    pred   = clean[xcol].values.astype(float)
    resid  = pred - actual

    color = PALETTE.get(target, '#333333')
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(actual, resid, alpha=0.35, s=18, color=color)
    ax.axhline(0, color="black", lw=1.4, linestyle="--")
    ax.set_xlabel(f"Actual {target}  (2025)", fontsize=10)
    ax.set_ylabel(f"Residual  (x{target} − {target})", fontsize=10)
    ax.set_title(f"2025 Residuals: {target}", fontsize=11, fontweight="bold", color=color)
    ax.grid(True, alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()

    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return buf


def fig_2025_percent_error(val_pred_df):
    """Distribution of percent errors across all targets for 2025."""
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = axes.flatten()
    targets = REGRESSION_TARGETS + SB_CS_TARGETS + ['MB']

    for i, target in enumerate(targets):
        xcol  = f"x{target}"
        ax    = axes[i]
        if target not in val_pred_df.columns or xcol not in val_pred_df.columns:
            ax.set_visible(False)
            continue

        clean  = val_pred_df[[target, xcol]].dropna()
        actual = clean[target].values.astype(float)
        pred   = clean[xcol].values.astype(float)
        denom  = np.where(np.abs(actual) < 0.01, np.nan, actual)
        pe     = (pred - actual) / denom * 100
        pe     = pe[~np.isnan(pe)]

        color = PALETTE.get(target, '#555555')
        ax.hist(pe, bins=30, color=color, edgecolor='white', alpha=0.8)
        ax.axvline(0,           color='black',   lw=1.3, linestyle='--')
        ax.axvline(np.mean(pe), color='#E74C3C', lw=1.3,
                   label=f"Mean {np.mean(pe):.1f}%")
        ax.set_title(f"{target}  (n={len(pe)})", fontsize=9, fontweight='bold', color=color)
        ax.set_xlabel("% Error", fontsize=8)
        ax.set_ylabel("Count", fontsize=8)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("2025 Out-of-Sample: Percent Error Distributions", fontsize=13, fontweight='bold')
    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return buf


def fig_2025_summary_bar(metrics_2025):
    """Bar chart comparing MAPE across all targets."""
    targets = [t for t in (REGRESSION_TARGETS + SB_CS_TARGETS + ['MB']) if t in metrics_2025]
    mapes   = [metrics_2025[t]['mean_ape'] for t in targets]
    r2s     = [metrics_2025[t]['r2']       for t in targets]
    colors_ = [PALETTE.get(t, '#555555')   for t in targets]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    ax1.bar(targets, mapes, color=colors_, edgecolor='white', alpha=0.85)
    ax1.set_ylabel("Mean Absolute % Error", fontsize=10)
    ax1.set_title("2025 Prediction Error by Target", fontsize=11, fontweight='bold')
    ax1.axhline(10, color='grey', lw=1, linestyle='--', label='10% threshold')
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3, axis='y')
    ax1.spines[["top", "right"]].set_visible(False)

    ax2.bar(targets, r2s, color=colors_, edgecolor='white', alpha=0.85)
    ax2.set_ylabel("R² (2025 out-of-sample)", fontsize=10)
    ax2.set_title("2025 R² by Target", fontsize=11, fontweight='bold')
    ax2.axhline(0.8, color='grey', lw=1, linestyle='--', label='R²=0.8 threshold')
    ax2.legend(fontsize=8)
    ax2.set_ylim(0, 1.05)
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.spines[["top", "right"]].set_visible(False)

    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return buf

In [10]:
# =============================================================================
# MB / xMB analysis (Bayesian — same as before, now on 2025 data)
# =============================================================================

def run_mb_analysis(df):
    clean = df[["Name", "Season", "MB", "xMB"]].dropna()
    clean = clean[clean["MB"] != 0].copy()
    n  = len(clean)
    y  = clean["MB"].values.astype(float)
    xhat = clean["xMB"].values.astype(float)

    clean["Percent_Error"]     = ((xhat - y) / np.abs(y)) * 100
    clean["Abs_Percent_Error"] = clean["Percent_Error"].abs()
    pe  = clean["Percent_Error"]
    ape = clean["Abs_Percent_Error"]

    pe_stats = {
        "n":             n,
        "mean_pe":       round(pe.mean(),    2),
        "median_pe":     round(pe.median(),  2),
        "mean_ape":      round(ape.mean(),   2),
        "median_ape":    round(ape.median(), 2),
        "std_pe":        round(pe.std(),     2),
        "min_pe":        round(pe.min(),     2),
        "max_pe":        round(pe.max(),     2),
        "pct_within_5":  round((ape <= 5).mean()  * 100, 1),
        "pct_within_10": round((ape <= 10).mean() * 100, 1),
        "pct_within_20": round((ape <= 20).mean() * 100, 1),
        "top_over":      clean.nlargest(5,  "Percent_Error"),
        "top_under":     clean.nsmallest(5, "Percent_Error"),
    }

    y_pred_0 = np.full_like(y, y.mean())
    rss_0    = np.sum((y - y_pred_0) ** 2)
    bic_0    = n * np.log(rss_0 / n) + 1 * np.log(n)

    X_des    = np.column_stack([np.ones(n), xhat])
    beta, _, _, _ = np.linalg.lstsq(X_des, y, rcond=None)
    y_pred_1 = X_des @ beta
    rss_1    = np.sum((y - y_pred_1) ** 2)
    bic_1    = n * np.log(rss_1 / n) + 2 * np.log(n)

    delta_bic    = bic_0 - bic_1
    bayes_factor = np.exp(delta_bic / 2)
    posterior_m1 = bayes_factor / (1 + bayes_factor)
    ss_tot       = np.sum((y - y.mean()) ** 2)
    r2_m1        = 1 - rss_1 / ss_tot

    def jeffreys_label(bf):
        if bf < 1:     return "Against Model 1 (xMB adds no predictive value)"
        elif bf < 3:   return "Anecdotal evidence for Model 1"
        elif bf < 10:  return "Moderate evidence for Model 1"
        elif bf < 30:  return "Strong evidence for Model 1"
        elif bf < 100: return "Very Strong evidence for Model 1"
        else:          return "Decisive evidence for Model 1"

    bayes_stats = {
        "bic_0":        round(bic_0,         2),
        "bic_1":        round(bic_1,         2),
        "delta_bic":    round(delta_bic,     2),
        "bayes_factor": round(bayes_factor,  2),
        "log10_bf":     round(np.log10(max(bayes_factor, 1e-10)), 2),
        "posterior_m1": round(posterior_m1 * 100, 1),
        "intercept":    round(beta[0],       3),
        "slope":        round(beta[1],       3),
        "r2_m1":        round(r2_m1,         4),
        "label":        jeffreys_label(bayes_factor),
    }
    return clean, pe_stats, bayes_stats

In [11]:
# =============================================================================
# MB / xMB FIGURES (same logic, applied to 2025 data)
# =============================================================================

def fig_percent_error_dist(clean):
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    pe = clean["Percent_Error"]

    ax = axes[0]
    ax.hist(pe, bins=40, color="#2980B9", edgecolor="white", alpha=0.85)
    ax.axvline(0,           color="black",   lw=1.5, linestyle="--", label="Zero error")
    ax.axvline(pe.mean(),   color="#E74C3C", lw=1.5, label=f"Mean ({pe.mean():.1f}%)")
    ax.axvline(pe.median(), color="#27AE60", lw=1.5, label=f"Median ({pe.median():.1f}%)")
    ax.set_xlabel("Percent Error (%)", fontsize=10)
    ax.set_ylabel("Count", fontsize=10)
    ax.set_title("MB Percent Error Distribution (2025)", fontsize=11, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

    ax2 = axes[1]
    # only 1 season, show player name top over/under
    ax2.scatter(clean["MB"], clean["xMB"], alpha=0.4, s=20, color="#2980B9")
    lim = [min(clean["MB"].min(), clean["xMB"].min()),
           max(clean["MB"].max(), clean["xMB"].max())]
    ax2.plot(lim, lim, color="grey", lw=1.2, linestyle="--")
    ax2.set_xlabel("Actual MB (2025)", fontsize=10)
    ax2.set_ylabel("Predicted xMB (2025)", fontsize=10)
    ax2.set_title("Predicted vs Actual MB (2025)", fontsize=11, fontweight="bold")
    ax2.grid(True, alpha=0.3)
    ax2.spines[["top", "right"]].set_visible(False)

    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return buf


def fig_mb_vs_xmb(clean):
    y    = clean["MB"].values.astype(float)
    xhat = clean["xMB"].values.astype(float)
    n    = len(clean)
    X_des = np.column_stack([np.ones(n), xhat])
    beta, _, _, _ = np.linalg.lstsq(X_des, y, rcond=None)
    x_line = np.linspace(xhat.min(), xhat.max(), 200)
    y_line = beta[0] + beta[1] * x_line

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(xhat, y, alpha=0.25, s=18, color="#2980B9", label="2025 Players")
    ax.plot(x_line, y_line, color="#E74C3C", lw=2,
            label=f"OLS fit  (slope={beta[1]:.3f}, intercept={beta[0]:.3f})")
    ax.plot(x_line, x_line, color="grey", lw=1.2, linestyle="--", label="Perfect (y=x)")
    ax.set_xlabel("xMB (Predicted from 2019-2024 model)", fontsize=11)
    ax.set_ylabel("MB (Actual 2025)", fontsize=11)
    ax.set_title("MB vs xMB — 2025 Out-of-Sample", fontsize=12, fontweight="bold", color="#2C3E50")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return buf


def fig_residuals(clean):
    y     = clean["MB"].values.astype(float)
    xhat  = clean["xMB"].values.astype(float)
    resid = xhat - y

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(y, resid, alpha=0.25, s=15, color="#8E44AD")
    ax.axhline(0, color="black", lw=1.4, linestyle="--")
    ax.set_xlabel("Actual MB (2025)", fontsize=11)
    ax.set_ylabel("Residual  (xMB − MB)", fontsize=11)
    ax.set_title("MB Residual Plot — 2025 Out-of-Sample", fontsize=12,
                 fontweight="bold", color="#2C3E50")
    ax.grid(True, alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    return buf

In [12]:
# =============================================================================
# PDF REPORT
# =============================================================================

def build_combined_pdf(all_results_reg, all_results_sb_cs, val_pred_df,
                       metrics_2025,
                       clean_mb, pe_stats, bayes_stats,
                       component_map=None,
                       output_path="statcast_2025_forecast_report.pdf"):

    doc = SimpleDocTemplate(
        output_path, pagesize=letter,
        leftMargin=0.75*inch, rightMargin=0.75*inch,
        topMargin=0.75*inch,  bottomMargin=0.75*inch
    )
    base = getSampleStyleSheet()
    S = {
        "title":   ParagraphStyle("T",    parent=base["Title"],
                                  fontSize=22, alignment=TA_CENTER, spaceAfter=6),
        "sub":     ParagraphStyle("Sub",  parent=base["Normal"],
                                  fontSize=11, alignment=TA_CENTER,
                                  textColor=colors.HexColor("#555555"), spaceAfter=4),
        "h1":      ParagraphStyle("H1",   parent=base["Heading1"],
                                  fontSize=14, textColor=colors.HexColor("#2C3E50"),
                                  spaceBefore=12, spaceAfter=6),
        "h2":      ParagraphStyle("H2",   parent=base["Heading2"],
                                  fontSize=11, textColor=colors.HexColor("#2980B9"),
                                  spaceBefore=8,  spaceAfter=4),
        "body":    ParagraphStyle("Body", parent=base["Normal"],
                                  fontSize=9.5, leading=14, spaceAfter=5),
        "small":   ParagraphStyle("Sm",   parent=base["Normal"],
                                  fontSize=8.5, leading=12,
                                  textColor=colors.HexColor("#444444")),
        "caption": ParagraphStyle("Cap",  parent=base["Normal"],
                                  fontSize=8,   leading=11,
                                  textColor=colors.grey,
                                  alignment=TA_CENTER, spaceAfter=8),
        "header":  ParagraphStyle("TH",   parent=base["Normal"],
                                  textColor=colors.white,
                                  fontName="Helvetica-Bold",
                                  fontSize=9, alignment=TA_CENTER),
        "verdict": ParagraphStyle("V",    parent=base["Normal"],
                                  fontSize=11, leading=16,
                                  textColor=colors.HexColor("#1a5276"),
                                  fontName="Helvetica-Bold",
                                  alignment=TA_CENTER, spaceAfter=8),
        "good":    ParagraphStyle("Good", parent=base["Normal"],
                                  fontSize=11, leading=16,
                                  textColor=colors.HexColor("#145a32"),
                                  fontName="Helvetica-Bold",
                                  alignment=TA_CENTER, spaceAfter=8),
    }

    story = []

    # ── Title page ──────────────────────────────────────────────────────────
    story.append(Spacer(1, 0.6*inch))
    story.append(Paragraph("Baseball Statcast", S["title"]))
    story.append(Paragraph("2025 Out-of-Sample Forecast Report", S["title"]))
    story.append(Spacer(1, 0.15*inch))
    story.append(divider())
    story.append(Spacer(1, 0.1*inch))
    story.append(Paragraph(
        "<b>Training period:</b> 2019–2024  |  <b>Validation period:</b> 2025<br/><br/>"
        "Models were trained exclusively on 2019–2024 data, then applied to 2025 players "
        "who were never seen during training. This report measures true out-of-sample accuracy "
        "using MAE, RMSE, R², and Percent Error — a more rigorous test than cross-validated "
        "R² on historical data.", S["body"]))
    story.append(PageBreak())

    # ── Part 1: Training-set model summary ──────────────────────────────────
    story.append(Paragraph("Part 1 — Training Model Summary (2019–2024)", S["h1"]))
    story.append(divider())
    story.append(Paragraph(
        "Models were selected and tuned using 2019–2024 data only. "
        "The best model per target (by held-out test R²) was retained for 2025 deployment.", S["body"]))

    summary_rows = [[th("Target", S), th("Best Model", S), th("Train R²", S),
                     th("CV R²", S), th("Train RMSE", S)]]
    for res in all_results_reg:
        if not res["models"]:
            continue
        best_name, best_m = max(res["models"].items(), key=lambda x: x[1]["r2"])
        summary_rows.append([res["target"], best_name, f"{best_m['r2']:.4f}",
                             f"{best_m['cv_r2_mean']:.4f}±{best_m['cv_r2_std']:.4f}",
                             f"{best_m['rmse']:.4f}"])
    for target, data in all_results_sb_cs.items():
        valid = [r for r in data["results"] if not np.isnan(r.get("R2", np.nan))]
        if not valid:
            continue
        best = max(valid, key=lambda r: r["R2"])
        summary_rows.append([target, best["Model"], str(best["R2"]),
                             str(best.get("AdjR2", "N/A")), str(best["RMSE"])])
    story.append(styled_table(summary_rows,
        [0.6*inch, 2.1*inch, 0.75*inch, 1.75*inch, 0.85*inch]))
    story.append(PageBreak())

    # ── Part 2: 2025 Out-of-Sample Validation ───────────────────────────────
    story.append(Paragraph("Part 2 — 2025 Out-of-Sample Validation", S["h1"]))
    story.append(divider())
    story.append(Paragraph(
        "The trained models were applied to 2025 batters. The table below shows how well "
        "the predictions match the actual 2025 statistics. R² here is <i>not</i> a training "
        "metric — it reflects genuine predictive accuracy on unseen data.", S["body"]))
    story.append(Spacer(1, 0.1*inch))

    val_rows = [[th("Target", S), th("n", S), th("R² (2025)", S),
                 th("MAE", S), th("RMSE", S),
                 th("Mean APE%", S), th("±5%", S), th("±10%", S), th("±20%", S)]]
    for target in REGRESSION_TARGETS + SB_CS_TARGETS + ['MB']:
        if target not in metrics_2025:
            continue
        m = metrics_2025[target]
        val_rows.append([
            target, str(m['n']),
            f"{m['r2']:.4f}", f"{m['mae']:.3f}", f"{m['rmse']:.3f}",
            f"{m['mean_ape']:.1f}%",
            f"{m['pct_within_5']}%",
            f"{m['pct_within_10']}%",
            f"{m['pct_within_20']}%",
        ])
    story.append(styled_table(val_rows,
        [0.55*inch, 0.4*inch, 0.75*inch, 0.65*inch,
         0.65*inch, 0.75*inch, 0.65*inch, 0.65*inch, 0.65*inch]))
    story.append(Spacer(1, 0.15*inch))

    story.append(Paragraph("Summary Overview", S["h2"]))
    buf_bar = fig_2025_summary_bar(metrics_2025)
    story.append(rl_image(buf_bar, 6.5, fig_w=10, fig_h=4))
    story.append(Paragraph(
        "Left: Mean Absolute Percent Error per target. Lower is better. "
        "Right: Out-of-sample R² per target. Higher is better.", S["caption"]))
    story.append(Spacer(1, 0.1*inch))

    story.append(Paragraph("Percent Error Distributions", S["h2"]))
    buf_pe = fig_2025_percent_error(val_pred_df)
    story.append(rl_image(buf_pe, 6.8, fig_w=14, fig_h=7))
    story.append(Paragraph(
        "Percent error distributions for each target in 2025. "
        "A distribution centred on zero indicates unbiased predictions.", S["caption"]))
    story.append(PageBreak())

    # ── Per-target scatter + residual ───────────────────────────────────────
    story.append(Paragraph("Per-Target: Predicted vs Actual (2025)", S["h1"]))
    story.append(divider())

    for target in REGRESSION_TARGETS + SB_CS_TARGETS + ['MB']:
        if target not in metrics_2025:
            continue
        story.append(Paragraph(f"Target: {target}", S["h2"]))
        m = metrics_2025[target]
        story.append(Paragraph(
            f"n={m['n']}  |  R²={m['r2']}  |  MAE={m['mae']}  |  RMSE={m['rmse']}  "
            f"|  Mean APE={m['mean_ape']}%", S["small"]))
        story.append(Spacer(1, 0.08*inch))

        buf_sc = fig_2025_scatter(val_pred_df, target)
        buf_re = fig_2025_residuals(val_pred_df, target)
        if buf_sc:
            story.append(rl_image(buf_sc, 5.5, fig_w=6, fig_h=5))
        if buf_re:
            story.append(rl_image(buf_re, 5.5, fig_w=6, fig_h=4))
        story.append(PageBreak())

    # ── Part 3: Player-level preview ────────────────────────────────────────
    story.append(Paragraph("Part 3 — 2025 Player-Level Predictions vs Actuals", S["h1"]))
    story.append(divider())
    story.append(Paragraph(
        "First 30 rows of the 2025 deployment. Full results saved to "
        "<b>expected_2025_stats.csv</b>.", S["body"]))

    disp_cols = [c for c in
        ["Name", "Season", "TB", "xTB", "BB", "xBB", "AB", "xAB",
         "H", "xH", "SB", "xSB", "CS", "xCS", "MB", "xMB"]
        if c in val_pred_df.columns]
    preview = val_pred_df[disp_cols].dropna(subset=["Name"]).head(30).round(2)
    col_w   = [1.3*inch, 0.45*inch] + [0.52*inch] * (len(disp_cols) - 2)
    dep_rows = [[th(c, S) for c in disp_cols]]
    for _, row in preview.iterrows():
        dep_rows.append([str(row[c]) for c in disp_cols])
    story.append(styled_table(dep_rows, col_w))
    story.append(PageBreak())

    # ── Part 4: MB Bayesian analysis (2025) ─────────────────────────────────
    story.append(Paragraph("Part 4 — MB vs xMB: Bayesian Comparison (2025 Data)", S["h1"]))
    story.append(divider())
    story.append(Paragraph(
        "The Bayesian comparison now uses <b>2025 actual MB</b> vs <b>xMB predicted by the "
        "2019–2024 model</b>. This is a true out-of-sample test of whether xMB carries "
        "meaningful predictive signal about MB in an unseen season.", S["body"]))
    story.append(Spacer(1, 0.1*inch))

    # ── xMB component selection table ───────────────────────────────────────
    story.append(Paragraph("xMB Component Selection", S["h2"]))
    story.append(Paragraph(
        f"Each of the 6 component stats is evaluated against the R² threshold of "
        f"<b>{XMB_R2_THRESHOLD}</b>. If a component's best training R² meets the threshold, "
        f"the <i>expected</i> (x&lt;stat&gt;) value is used in the xMB formula; "
        f"otherwise the <i>actual observed</i> stat is used.", S["body"]))
    story.append(Spacer(1, 0.08*inch))

    if component_map:
        comp_rows = [[th("Component", S), th("Training R²", S),
                      th(f"Threshold (≥{XMB_R2_THRESHOLD})", S), th("Used in xMB", S)]]
        for stat in XMB_COMPONENTS:
            info    = component_map.get(stat, {})
            r2_val  = info.get('r2', 0.0)
            use_col = info.get('col', stat)
            passed  = r2_val >= XMB_R2_THRESHOLD
            comp_rows.append([
                stat,
                f"{r2_val:.4f}",
                "✓ Pass" if passed else "✗ Fail",
                use_col,
            ])
        story.append(styled_table(comp_rows,
            [1.0*inch, 1.1*inch, 1.4*inch, 1.4*inch],
            header_color="#2C3E50"))
        story.append(Paragraph(
            "Components marked ✗ Fail use the real observed stat in the xMB formula.",
            S["caption"]))
    story.append(Spacer(1, 0.15*inch))

    sum_rows = [
        [th("Metric", S), th("Value", S)],
        ["Players analysed",         str(pe_stats["n"])],
        ["Mean Percent Error",        f"{pe_stats['mean_pe']}%  (+ = over-predict)"],
        ["Median Percent Error",      f"{pe_stats['median_pe']}%"],
        ["Mean Abs. Percent Error",   f"{pe_stats['mean_ape']}%"],
        ["Median Abs. Percent Error", f"{pe_stats['median_ape']}%"],
        ["Std Dev of Percent Error",  f"{pe_stats['std_pe']}%"],
        ["Min / Max Percent Error",   f"{pe_stats['min_pe']}% / {pe_stats['max_pe']}%"],
        ["Players within ±5%",        f"{pe_stats['pct_within_5']}%"],
        ["Players within ±10%",       f"{pe_stats['pct_within_10']}%"],
        ["Players within ±20%",       f"{pe_stats['pct_within_20']}%"],
    ]
    story.append(styled_table(sum_rows, [3.2*inch, 3.0*inch], header_color="#2C3E50"))
    story.append(Spacer(1, 0.15*inch))

    buf_dist = fig_percent_error_dist(clean_mb)
    story.append(rl_image(buf_dist, 6.5, fig_w=9, fig_h=4))
    story.append(Paragraph(
        "Left: distribution of MB percent error in 2025. Right: xMB vs MB scatter.", S["caption"]))
    story.append(Spacer(1, 0.1*inch))

    story.append(Paragraph("Top 5 Over-Predicted  (xMB >> MB)", S["h2"]))
    over = pe_stats["top_over"][["Name", "Season", "MB", "xMB", "Percent_Error"]].round(2)
    over_rows = [[th(c, S) for c in ["Name", "Season", "MB", "xMB", "% Error"]]]
    for _, row in over.iterrows():
        over_rows.append([str(row["Name"]), str(int(row["Season"])),
                          str(row["MB"]), str(row["xMB"]), f"{row['Percent_Error']:.1f}%"])
    story.append(styled_table(over_rows,
        [2.2*inch, 0.8*inch, 0.9*inch, 0.9*inch, 1.0*inch], header_color="#C0392B"))
    story.append(Spacer(1, 0.1*inch))

    story.append(Paragraph("Top 5 Under-Predicted  (xMB << MB)", S["h2"]))
    under = pe_stats["top_under"][["Name", "Season", "MB", "xMB", "Percent_Error"]].round(2)
    under_rows = [[th(c, S) for c in ["Name", "Season", "MB", "xMB", "% Error"]]]
    for _, row in under.iterrows():
        under_rows.append([str(row["Name"]), str(int(row["Season"])),
                           str(row["MB"]), str(row["xMB"]), f"{row['Percent_Error']:.1f}%"])
    story.append(styled_table(under_rows,
        [2.2*inch, 0.8*inch, 0.9*inch, 0.9*inch, 1.0*inch], header_color="#1A5276"))
    story.append(PageBreak())

    story.append(Paragraph("Bayesian Model Comparison (2025)", S["h2"]))
    bf_rows = [
        [th("Statistic", S), th("Value", S)],
        ["BIC — Model 0 (Null)",                    str(bayes_stats["bic_0"])],
        ["BIC — Model 1 (xMB predictor)",           str(bayes_stats["bic_1"])],
        ["Delta BIC (BIC₀ − BIC₁)",                 str(bayes_stats["delta_bic"])],
        ["Bayes Factor (BF₁₀)",                     str(bayes_stats["bayes_factor"])],
        ["log₁₀ Bayes Factor",                      str(bayes_stats["log10_bf"])],
        ["Posterior P(Model 1 | 2025 data)",        f"{bayes_stats['posterior_m1']}%"],
        ["OLS Intercept (a)",                        str(bayes_stats["intercept"])],
        ["OLS Slope (b)",                            str(bayes_stats["slope"])],
        ["Model 1 R²",                               str(bayes_stats["r2_m1"])],
    ]
    story.append(styled_table(bf_rows, [3.2*inch, 3.0*inch], header_color="#2980B9"))
    story.append(Spacer(1, 0.1*inch))
    story.append(Paragraph(f"Verdict (Jeffreys' Scale):  {bayes_stats['label']}", S["verdict"]))
    story.append(Spacer(1, 0.15*inch))

    story.append(rl_image(fig_mb_vs_xmb(clean_mb), 6.0, fig_w=7, fig_h=5))
    story.append(Paragraph(
        "Scatter of 2025 actual MB vs model-predicted xMB. "
        "Slope near 1 and intercept near 0 = good calibration.", S["caption"]))
    story.append(Spacer(1, 0.1*inch))
    story.append(rl_image(fig_residuals(clean_mb), 6.0, fig_w=7, fig_h=4))
    story.append(Paragraph(
        "Residual plot: xMB − MB vs actual MB for 2025.", S["caption"]))

    doc.build(story)
    print(f"\nPDF saved to: {output_path}")

In [ ]:
# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":

    # ── 1. Train regression models on 2019-2024 ──────────────────────────
    print("\n" + "="*60)
    print("  TRAINING: Regression models on 2019–2024")
    print("="*60)
    all_results_reg = []
    for target in REGRESSION_TARGETS:
        result = run_regression_analysis(target, BASE_FEATURES, train_df)
        all_results_reg.append(result)

    # ── 2. Train SB/CS models on 2019-2024 ───────────────────────────────
    print("\n" + "="*60)
    print("  TRAINING: SB / CS models on 2019–2024")
    print("="*60)
    all_results_sb_cs = {}
    for target in SB_CS_TARGETS:
        print(f"\nFitting models for {target}…")
        pack = fit_sb_cs_models(train_df, target)
        results, primary, x, y, x_plot, curves = pack

        if not results:
            all_results_sb_cs[target] = {"results": [], "primary": None}
            continue

        scatter_buf = make_scatter_plot(x, y, results, x_plot, curves, primary, target)
        resid_buf   = make_residual_plot(x, y, results, primary, target)

        all_results_sb_cs[target] = {
            "results":     results,
            "primary":     primary,
            "scatter_buf": scatter_buf,
            "resid_buf":   resid_buf,
        }

    # ── 3. Deploy models to 2025 data ────────────────────────────────────
    print("\n" + "="*60)
    print("  DEPLOYING to 2025 data")
    print("="*60)
    val_pred_df = deploy_to_2025(val_df, all_results_reg, all_results_sb_cs)

    # ── 4. Decide which components to use in xMB ─────────────────────────
    # For each of the 6 MB components, use x<stat> only if its model R² >= 0.5,
    # otherwise fall back to the real observed stat.
    print("\n" + "="*60)
    print(f"  xMB COMPONENT SELECTION  (threshold R² ≥ {XMB_R2_THRESHOLD})")
    print("="*60)
    component_map = select_xmb_components(all_results_reg, all_results_sb_cs)

    # ── 5. Compute 2025 MB / xMB using selected components ───────────────
    val_pred_df = compute_mb(val_pred_df, component_map=component_map)

    # Save 2025 predictions CSV
    preview_cols = [c for c in
        ["Name", "Season", "TB", "xTB", "BB", "xBB", "AB", "xAB",
         "H", "xH", "SB", "xSB", "CS", "xCS", "MB", "xMB"]
        if c in val_pred_df.columns]
    val_pred_df[preview_cols].to_csv("expected_2025_stats.csv", index=False)
    print("Saved: expected_2025_stats.csv")

    # ── 6. Out-of-sample validation metrics ──────────────────────────────
    print("\n" + "="*60)
    print("  2025 OUT-OF-SAMPLE VALIDATION METRICS")
    print("="*60)
    metrics_2025 = run_2025_validation(val_pred_df)

    # ── 7. MB Bayesian comparison on 2025 data ───────────────────────────
    clean_mb, pe_stats, bayes_stats = run_mb_analysis(val_pred_df)
    clean_mb[["Name", "Season", "MB", "xMB", "Percent_Error", "Abs_Percent_Error"]].to_csv(
        "mb_2025_comparison.csv", index=False)
    print("Saved: mb_2025_comparison.csv")

    # ── 8. Build PDF ──────────────────────────────────────────────────────
    OUTPUT_PDF = "statcast_2025_forecast_report.pdf"
    build_combined_pdf(
        all_results_reg, all_results_sb_cs, val_pred_df,
        metrics_2025,
        clean_mb, pe_stats, bayes_stats,
        component_map=component_map,
        output_path=OUTPUT_PDF
    )

    try:
        os.startfile(OUTPUT_PDF)
    except Exception:
        pass   


  TRAINING: Regression models on 2019–2024

  TARGET: TB
Selected features (Linear/Ridge): ['Barrel%', 'maxEV', 'HardHit', 'HardHit%', 'CSW%', 'EV', 'Pitches']
All features (Lasso/RF): ['O-Swing%', 'Z-Swing%', 'Swing%', 'O-Contact%', 'Z-Contact%', 'Contact%', 'Zone%', 'F-Strike%', 'SwStr%', 'Barrel%', 'maxEV', 'HardHit', 'HardHit%', 'CStr%', 'CSW%', 'EV', 'LA', 'bolts', 'hp_to_1b', 'sprint_speed', 'Pitches', 'LD%', 'GB%']
  Linear Regression         R²=0.9541  CV R²=0.9484  RMSE=17.1072
  Ridge Regression          R²=0.9541  CV R²=0.9484  RMSE=17.1072
  Lasso Regression          R²=0.9535  CV R²=0.9502  RMSE=18.2171
  Random Forest             R²=0.9369  CV R²=0.9251  RMSE=21.2225
  → Deployed: 'Linear Regression'

  TARGET: BB
Selected features (Linear/Ridge): ['O-Swing%', 'Swing%', 'F-Strike%', 'Barrel%', 'maxEV', 'HardHit', 'HardHit%', 'EV', 'Pitches']
All features (Lasso/RF): ['O-Swing%', 'Z-Swing%', 'Swing%', 'O-Contact%', 'Z-Contact%', 'Contact%', 'Zone%', 'F-Strike%', 'SwStr%',

In [14]:
# Must be other ways to present the models (Why a model works compared to another one)

In [15]:
# Look to understand the math of the current models
# Then look at frisbee models
# Eventually get to 3D models (Shapes of vectors)